# 05 — Hyperparameter tuning

**Why:** the midterm models used hand-picked settings. Here every model is tuned with Optuna (TPE sampler, seed 42) on the **2020 validation year only**: train 2014–2019, score 2020. The 2021–2022 test years are not touched.

**Objectives:** severity regression minimizes RMSE of log(1 + % out); severity classification maximizes macro-F1 (accuracy is recorded too); restoration maximizes the concordance index.

The winning settings are copied into `models_severity.py` and `models_survival.py`; notebooks 02–04 then refit with them and report test results. Full runs take about 1.5 h on 8 CPU cores; lower `N` for a quick check.

In [ ]:
# Works on Colab (Drive) or locally (repo checkout)
import os, sys, pathlib, warnings
warnings.filterwarnings("ignore")
try:
    if os.environ.get("LIGHTSOUT_PROJECT"):  # a repo checkout, e.g. run with nbconvert inside Colab
        raise ImportError
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = pathlib.Path("/content/drive/MyDrive/LightsOut")
    ZIP = pathlib.Path("/content/drive/MyDrive/LightsOut_code.zip")
    if ZIP.exists():  # refresh code (src/, notebooks/) from the latest uploaded zip; data/ is untouched
        import zipfile; zipfile.ZipFile(ZIP).extractall(PROJECT)
    !pip -q install polars lifelines scikit-survival shap geopandas lightgbm optuna
except ImportError:
    PROJECT = pathlib.Path(os.environ["LIGHTSOUT_PROJECT"]) if os.environ.get("LIGHTSOUT_PROJECT") else \
        pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(PROJECT / "src"))
os.chdir(PROJECT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from lightsout import config as C, viz
viz.setup()
df = pd.read_parquet(C.PROCESSED / "model_table.parquet")
print(f"{len(df):,} (storm episode, county) units, {df.fips.nunique():,} counties, years {df.year.min()}-{df.year.max()}")

Rolling-origin validation: every candidate is trained on the years before 2019 (and 2020) and scored on that year; the two scores are averaged. History features are recomputed inside each fold. The test years 2021-2022 are never used. This takes several hours on a laptop; `python -m lightsout.tuning` runs the same thing from the terminal.

In [ ]:
from lightsout import tuning as T
N = {"lgb_reg": 40, "lgb_clf": 30, "rf_reg": 20, "rf_clf": 20, "rsf": 15, "lgb_surv": 30}
fd = T.folds(df)
pd.DataFrame([(m, k, v) for m, sp in T.SPACES.items() for k, v in sp.items()],
             columns=["model", "hyperparameter", "search range"]).set_index(["model", "hyperparameter"])

In [ ]:
lin = T.tune_linear(df, fd); cox = T.tune_cox(df, fd)
display(lin); cox.pivot(index="penalizer", columns="l1_ratio", values="val C-index")

In [ ]:
studies = {name: T.tune_severity(df, name, N[name], fd) for name in ["lgb_reg", "lgb_clf", "rf_reg", "rf_clf"]}
studies["rsf"] = T.tune_rsf(df, N["rsf"], fd)
studies["lgb_surv"] = T.tune_lgb_surv(df, N["lgb_surv"], fd)

In [ ]:
best = T.summary(studies)
best.to_csv(C.REPORTS / "tuning_best.csv", index=False)
best.set_index("search").T